# Traditional Species Classification and Robustness

This notebook compares **HOG + Linear SVM**, **HSV histogram + Linear SVM**, and **HOG + HSV + Linear SVM** on the 500-class iNaturalist subset. It then keeps each trained model fixed and evaluates Gaussian noise, Gaussian blur, brightness reduction, and JPEG compression at four severity levels.

## 1. Install dependencies

In [ ]:
%pip install -q opencv-python-headless scikit-image scikit-learn joblib tqdm matplotlib seaborn pandas

## 2. Imports and experiment constants

In [ ]:
from __future__ import annotations

import argparse
import csv
import json
import time
from pathlib import Path
from types import SimpleNamespace

import cv2
import joblib
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    precision_recall_fscore_support,
    top_k_accuracy_score,
)
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import MaxAbsScaler
from sklearn.svm import LinearSVC
from skimage.feature import hog
from tqdm.auto import tqdm

IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
VARIANTS = ("hog", "hsv", "combined")
DEGRADATIONS = {
    "gaussian_noise": [10, 20, 30, 40],
    "gaussian_blur": [3, 5, 7, 9],
    "brightness": [0.8, 0.6, 0.4, 0.2],
    "jpeg": [70, 50, 30, 10],
}

## 3. Configuration
Change `DATA_DIR` to the folder containing `train`, `val`, and `test`.

In [ ]:
# Change only these paths/settings before running the notebook.
DATA_DIR = Path("/content/dataset")
OUTPUT_DIR = Path("/content/traditional_outputs")

args = SimpleNamespace(
    data_dir=DATA_DIR,
    output_dir=OUTPUT_DIR,
    image_size=128,
    hsv_bins=8,
    c_values=[0.1, 1.0],
    max_iter=5000,
    seed=42,
    force_features=False,
    skip_confusion_matrix=False,
    run_robustness=True,
)

print("Dataset:", args.data_dir)
print("Outputs:", args.output_dir)

## 4. Verify dataset structure

In [ ]:
# Expected: train/<class>/images, val/<class>/images, test/<class>/images
for split in ("train", "val", "test"):
    split_dir = args.data_dir / split
    if not split_dir.is_dir():
        raise FileNotFoundError(f"Missing split directory: {split_dir}")
    class_count = sum(path.is_dir() for path in split_dir.iterdir())
    image_count = sum(
        path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS
        for path in split_dir.rglob("*")
    )
    print(f"{split:5s}: {class_count} classes, {image_count} images")

## 5. Feature, training, evaluation, caching, and robustness functions

In [ ]:
def discover_classes(train_dir: Path) -> list[str]:
    classes = sorted(p.name for p in train_dir.iterdir() if p.is_dir())
    if not classes:
        raise ValueError(f"No class folders found inside {train_dir}")
    return classes


def collect_images(split_dir: Path, classes: list[str]) -> tuple[list[Path], np.ndarray]:
    paths: list[Path] = []
    labels: list[int] = []
    for class_id, class_name in enumerate(classes):
        class_dir = split_dir / class_name
        if not class_dir.is_dir():
            raise ValueError(f"Missing class folder: {class_dir}")
        class_images = sorted(
            p for p in class_dir.rglob("*")
            if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS
        )
        if not class_images:
            raise ValueError(f"No images found in {class_dir}")
        paths.extend(class_images)
        labels.extend([class_id] * len(class_images))
    return paths, np.asarray(labels, dtype=np.int32)


def extract_hog(image_rgb: np.ndarray) -> np.ndarray:
    return hog(
        image_rgb,
        orientations=9,
        pixels_per_cell=(16, 16),
        cells_per_block=(2, 2),
        block_norm="L2-Hys",
        channel_axis=-1,
        feature_vector=True,
    ).astype(np.float32)


def extract_hsv_histogram(image_bgr: np.ndarray, bins: int) -> np.ndarray:
    hsv = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2HSV)
    histogram = cv2.calcHist(
        [hsv], [0, 1, 2], None, [bins, bins, bins], [0, 180, 0, 256, 0, 256]
    ).flatten().astype(np.float32)
    histogram /= histogram.sum() + 1e-12
    return histogram


def extract_features(
    paths: list[Path],
    image_size: int,
    hsv_bins: int,
    description: str,
    degradation: str | None = None,
    severity: float | None = None,
    seed: int = 42,
) -> tuple[np.ndarray, np.ndarray]:
    hog_features: list[np.ndarray] = []
    hsv_features: list[np.ndarray] = []
    for image_index, path in enumerate(tqdm(paths, desc=f"Extracting {description}", unit="image")):
        image_bgr = cv2.imread(str(path), cv2.IMREAD_COLOR)
        if image_bgr is None:
            raise ValueError(f"Could not read image: {path}")
        image_bgr = cv2.resize(
            image_bgr, (image_size, image_size), interpolation=cv2.INTER_AREA
        )
        if degradation is not None and severity is not None:
            image_bgr = degrade_image(
                image_bgr, degradation, severity, seed + image_index
            )
        image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
        hog_features.append(extract_hog(image_rgb))
        hsv_features.append(extract_hsv_histogram(image_bgr, hsv_bins))
    return np.stack(hog_features), np.stack(hsv_features)


def degrade_image(
    image_bgr: np.ndarray, degradation: str, severity: float, seed: int
) -> np.ndarray:
    """Apply one deterministic degradation to an already resized test image."""
    if degradation == "gaussian_noise":
        rng = np.random.default_rng(seed)
        noise = rng.normal(0.0, severity, image_bgr.shape)
        return np.clip(image_bgr.astype(np.float32) + noise, 0, 255).astype(np.uint8)
    if degradation == "gaussian_blur":
        kernel_size = int(severity)
        return cv2.GaussianBlur(image_bgr, (kernel_size, kernel_size), sigmaX=0)
    if degradation == "brightness":
        return np.clip(image_bgr.astype(np.float32) * severity, 0, 255).astype(np.uint8)
    if degradation == "jpeg":
        success, encoded = cv2.imencode(
            ".jpg", image_bgr, [cv2.IMWRITE_JPEG_QUALITY, int(severity)]
        )
        if not success:
            raise ValueError("JPEG degradation failed")
        decoded = cv2.imdecode(encoded, cv2.IMREAD_COLOR)
        if decoded is None:
            raise ValueError("JPEG degradation could not be decoded")
        return decoded
    raise ValueError(f"Unknown degradation: {degradation}")


def load_or_extract_split(
    split: str,
    paths: list[Path],
    labels: np.ndarray,
    cache_dir: Path,
    image_size: int,
    hsv_bins: int,
    force: bool,
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    cache_path = cache_dir / f"{split}_size{image_size}_bins{hsv_bins}.npz"
    if cache_path.exists() and not force:
        data = np.load(cache_path)
        if len(data["labels"]) != len(labels) or not np.array_equal(data["labels"], labels):
            raise ValueError(
                f"Cached labels do not match {split}. Delete {cache_path} or use --force-features."
            )
        return data["hog"], data["hsv"], data["labels"]

    hog_x, hsv_x = extract_features(paths, image_size, hsv_bins, split)
    np.savez_compressed(cache_path, hog=hog_x, hsv=hsv_x, labels=labels)
    return hog_x, hsv_x, labels


def select_features(variant: str, hog_x: np.ndarray, hsv_x: np.ndarray) -> np.ndarray:
    if variant == "hog":
        return hog_x
    if variant == "hsv":
        return hsv_x
    if variant == "combined":
        return np.concatenate((hog_x, hsv_x), axis=1)
    raise ValueError(f"Unknown variant: {variant}")


def calculate_metrics(
    y_true: np.ndarray, predictions: np.ndarray, scores: np.ndarray, labels: np.ndarray
) -> dict[str, float]:
    precision, recall, macro_f1, _ = precision_recall_fscore_support(
        y_true, predictions, average="macro", zero_division=0
    )
    k = min(5, len(labels))
    return {
        "accuracy": float(accuracy_score(y_true, predictions)),
        "top_1_accuracy": float(top_k_accuracy_score(y_true, scores, k=1, labels=labels)),
        "top_5_accuracy": float(top_k_accuracy_score(y_true, scores, k=k, labels=labels)),
        "macro_precision": float(precision),
        "macro_recall": float(recall),
        "macro_f1": float(macro_f1),
    }


def train_model(x_train: np.ndarray, y_train: np.ndarray, c_value: float, max_iter: int):
    model = make_pipeline(
        MaxAbsScaler(),
        LinearSVC(C=c_value, dual="auto", max_iter=max_iter, random_state=42),
    )
    start = time.perf_counter()
    model.fit(x_train, y_train)
    return model, time.perf_counter() - start


def evaluate_model(model, x: np.ndarray, y: np.ndarray, labels: np.ndarray):
    start = time.perf_counter()
    predictions = model.predict(x)
    scores = model.decision_function(x)
    elapsed = time.perf_counter() - start
    return calculate_metrics(y, predictions, scores, labels), predictions, elapsed


def save_confusion_matrix(
    y_true: np.ndarray, predictions: np.ndarray, output_path: Path
) -> None:
    matrix = confusion_matrix(y_true, predictions, normalize="true")
    plt.figure(figsize=(14, 12))
    sns.heatmap(matrix, cmap="Blues", xticklabels=False, yticklabels=False)
    plt.title("Normalized confusion matrix (all classes)")
    plt.xlabel("Predicted class")
    plt.ylabel("True class")
    plt.tight_layout()
    plt.savefig(output_path, dpi=200)
    plt.close()


def safe_severity_name(severity: float) -> str:
    return str(severity).replace(".", "p")


def load_or_extract_degraded_test(
    paths: list[Path],
    labels: np.ndarray,
    cache_dir: Path,
    image_size: int,
    hsv_bins: int,
    degradation: str,
    severity: float,
    seed: int,
    force: bool,
) -> tuple[np.ndarray, np.ndarray]:
    cache_path = cache_dir / (
        f"test_{degradation}_{safe_severity_name(severity)}_"
        f"size{image_size}_bins{hsv_bins}.npz"
    )
    if cache_path.exists() and not force:
        data = np.load(cache_path)
        if not np.array_equal(data["labels"], labels):
            raise ValueError(f"Cached labels do not match: {cache_path}")
        return data["hog"], data["hsv"]

    hog_x, hsv_x = extract_features(
        paths,
        image_size,
        hsv_bins,
        f"test {degradation} severity={severity}",
        degradation=degradation,
        severity=severity,
        seed=seed,
    )
    np.savez_compressed(cache_path, hog=hog_x, hsv=hsv_x, labels=labels)
    return hog_x, hsv_x


def save_robustness_plots(results: list[dict], output_dir: Path) -> None:
    for degradation, severity_values in DEGRADATIONS.items():
        degradation_rows = [r for r in results if r["degradation"] == degradation]
        for metric, label in (("top_1_accuracy", "Top-1 accuracy"), ("macro_f1", "Macro-F1")):
            plt.figure(figsize=(8, 5))
            for variant in VARIANTS:
                rows = [r for r in degradation_rows if r["variant"] == variant]
                clean = next(r for r in rows if r["severity_index"] == 0)
                degraded = sorted(
                    (r for r in rows if r["severity_index"] > 0),
                    key=lambda r: r["severity_index"],
                )
                y_values = [clean[metric]] + [r[metric] for r in degraded]
                plt.plot(range(5), y_values, marker="o", label=variant.upper())
            tick_labels = ["Clean"] + [str(value) for value in severity_values]
            plt.xticks(range(5), tick_labels)
            plt.xlabel("Degradation severity")
            plt.ylabel(label)
            plt.title(f"{label} under {degradation.replace('_', ' ')}")
            plt.grid(alpha=0.3)
            plt.legend()
            plt.tight_layout()
            plt.savefig(output_dir / f"robustness_{degradation}_{metric}.png", dpi=200)
            plt.close()


def run_robustness_study(
    models: dict,
    clean_test_metrics: dict,
    test_paths: list[Path],
    y_test: np.ndarray,
    labels: np.ndarray,
    args: argparse.Namespace,
    cache_dir: Path,
) -> list[dict]:
    robustness_results: list[dict] = []
    for degradation, severity_values in DEGRADATIONS.items():
        for variant in VARIANTS:
            robustness_results.append({
                "variant": variant,
                "degradation": degradation,
                "severity_index": 0,
                "severity_value": "clean",
                "top_1_accuracy": clean_test_metrics[variant]["top_1_accuracy"],
                "macro_f1": clean_test_metrics[variant]["macro_f1"],
                "performance_drop_top_1": 0.0,
                "performance_drop_macro_f1": 0.0,
            })

        for severity_index, severity in enumerate(severity_values, start=1):
            hog_x, hsv_x = load_or_extract_degraded_test(
                test_paths,
                y_test,
                cache_dir,
                args.image_size,
                args.hsv_bins,
                degradation,
                severity,
                args.seed,
                args.force_features,
            )
            for variant in VARIANTS:
                x_test = select_features(variant, hog_x, hsv_x)
                metrics, _, _ = evaluate_model(models[variant], x_test, y_test, labels)
                clean = clean_test_metrics[variant]
                row = {
                    "variant": variant,
                    "degradation": degradation,
                    "severity_index": severity_index,
                    "severity_value": severity,
                    "top_1_accuracy": metrics["top_1_accuracy"],
                    "macro_f1": metrics["macro_f1"],
                    "performance_drop_top_1": clean["top_1_accuracy"] - metrics["top_1_accuracy"],
                    "performance_drop_macro_f1": clean["macro_f1"] - metrics["macro_f1"],
                }
                robustness_results.append(row)
                print(json.dumps(row, indent=2))

    with (args.output_dir / "robustness_results.csv").open("w", newline="") as file:
        writer = csv.DictWriter(file, fieldnames=list(robustness_results[0].keys()))
        writer.writeheader()
        writer.writerows(robustness_results)
    (args.output_dir / "robustness_results.json").write_text(
        json.dumps(robustness_results, indent=2)
    )
    save_robustness_plots(robustness_results, args.output_dir)
    return robustness_results

## 6. Extract features and run the three ablations

The SVM regularization value is selected using **validation macro-F1**. The held-out test set is evaluated only after model selection.

In [ ]:
args.output_dir.mkdir(parents=True, exist_ok=True)
cache_dir = args.output_dir / "feature_cache"
model_dir = args.output_dir / "models"
cache_dir.mkdir(exist_ok=True)
model_dir.mkdir(exist_ok=True)

classes = discover_classes(args.data_dir / "train")
(args.output_dir / "classes.json").write_text(json.dumps(classes, indent=2))
print(f"Found {len(classes)} classes")

split_data = {}
split_paths = {}
for split in ("train", "val", "test"):
    paths, labels = collect_images(args.data_dir / split, classes)
    split_paths[split] = paths
    print(f"{split}: {len(paths)} images")
    split_data[split] = load_or_extract_split(
        split, paths, labels, cache_dir, args.image_size, args.hsv_bins,
        args.force_features,
    )

all_labels = np.arange(len(classes))
results = []
best_models = {}
clean_test_metrics = {}

for variant in VARIANTS:
    train_hog, train_hsv, y_train = split_data["train"]
    val_hog, val_hsv, y_val = split_data["val"]
    test_hog, test_hsv, y_test = split_data["test"]
    x_train = select_features(variant, train_hog, train_hsv)
    x_val = select_features(variant, val_hog, val_hsv)
    x_test = select_features(variant, test_hog, test_hsv)

    best = None
    for c_value in args.c_values:
        print(f"\nTraining {variant} with C={c_value}")
        model, training_seconds = train_model(x_train, y_train, c_value, args.max_iter)
        val_metrics, _, val_seconds = evaluate_model(model, x_val, y_val, all_labels)
        row = {
            "variant": variant,
            "C": c_value,
            "split": "validation",
            "training_seconds": training_seconds,
            "inference_seconds": val_seconds,
            **val_metrics,
        }
        results.append(row)
        print(row)
        if best is None or val_metrics["macro_f1"] > best[0]:
            best = (val_metrics["macro_f1"], c_value, model, training_seconds)

    _, best_c, best_model, training_seconds = best
    test_metrics, test_predictions, test_seconds = evaluate_model(
        best_model, x_test, y_test, all_labels
    )
    test_row = {
        "variant": variant,
        "C": best_c,
        "split": "test",
        "training_seconds": training_seconds,
        "inference_seconds": test_seconds,
        **test_metrics,
    }
    results.append(test_row)
    best_models[variant] = best_model
    clean_test_metrics[variant] = test_metrics
    print(f"\nFinal test result for {variant}:", test_row)
    joblib.dump(best_model, model_dir / f"{variant}_linear_svm.joblib")
    if not args.skip_confusion_matrix:
        save_confusion_matrix(
            y_test, test_predictions,
            args.output_dir / f"{variant}_confusion_matrix.png",
        )

with (args.output_dir / "results.csv").open("w", newline="") as file:
    writer = csv.DictWriter(file, fieldnames=list(results[0].keys()))
    writer.writeheader()
    writer.writerows(results)
(args.output_dir / "results.json").write_text(json.dumps(results, indent=2))

print("Clean-data experiments completed.")

## 7. Compare clean-test results

In [ ]:
import pandas as pd

clean_results_df = pd.DataFrame(results)
display(clean_results_df)

test_comparison = clean_results_df[clean_results_df["split"] == "test"].copy()
display(test_comparison[[
    "variant", "C", "top_1_accuracy", "top_5_accuracy",
    "macro_precision", "macro_recall", "macro_f1",
    "training_seconds", "inference_seconds",
]].sort_values("macro_f1", ascending=False))

## 8. Test-time robustness study

Only test images are degraded. The models are not retrained or fine-tuned. Each degradation has four severity levels, and results include clean baselines, top-1 accuracy, macro-F1, and absolute performance drops.

In [ ]:
robustness_results = run_robustness_study(
    best_models,
    clean_test_metrics,
    split_paths["test"],
    split_data["test"][2],
    all_labels,
    args,
    cache_dir,
)

robustness_df = pd.DataFrame(robustness_results)
display(robustness_df.head(15))
print("Robustness outputs saved to:", args.output_dir)

## 9. Robustness summary for the report

In [ ]:
worst_case_summary = (
    robustness_df[robustness_df["severity_index"] > 0]
    .groupby(["variant", "degradation"], as_index=False)
    .agg(
        worst_top1=("top_1_accuracy", "min"),
        worst_macro_f1=("macro_f1", "min"),
        maximum_top1_drop=("performance_drop_top_1", "max"),
        maximum_macro_f1_drop=("performance_drop_macro_f1", "max"),
    )
)
display(worst_case_summary)

print("Best clean model:")
display(test_comparison.nlargest(1, "macro_f1")[["variant", "top_1_accuracy", "macro_f1"]])

## Interpretation checklist

- Compare whether HOG, HSV, or the combined representation performs best on clean data.
- Explain whether blur/noise damages HOG by destroying or creating gradients.
- Explain whether brightness reduction damages HSV colour information.
- Identify the model with the smallest top-1 and macro-F1 drops.
- Discuss why the 500-class confusion matrix is supplemented by quantitative metrics.
- Report all settings, random seed, timing, and degradation severity values.